# M5 — Web provenance: scenario testing (notebook 06)

Tests M5's ability to find an AI image's web origin and inherit stronger
evidence from upstream copies. The core loop:

```
stripped image → inconclusive locally
    → reverse search → find upstream copy
    → re-analyze upstream → verified (C2PA intact)
    → verdict inherited through documented chain
```

**API key required:** Google Cloud Vision WEB_DETECTION (1,000 free/month).
Store your key in Colab Secrets (🔑 sidebar → `GOOGLE_CLOUD_API_KEY`).
The key never appears in the notebook or its outputs.

Convention: commit WITH outputs (but clear any cell that accidentally
prints the key — the `userdata.get` pattern prevents this).

In [1]:
# ── Setup ──────────────────────────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e . requests

import os, sys
sys.path.insert(0, "/content/AI-image-Checkers")

# Load API key from Colab Secrets (🔑 sidebar → GOOGLE_CLOUD_API_KEY)
from google.colab import userdata
os.environ["GOOGLE_CLOUD_API_KEY"] = userdata.get("GOOGLE_CLOUD_API_KEY")
print("API key loaded ✓" if os.environ.get("GOOGLE_CLOUD_API_KEY") else "⚠ no key found")

/content/AI-image-Checkers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 3.1 MB/s eta 0:00:00
  Building editable for ai-image-id (pyproject.toml) ... done
API key loaded ✓


## §1 — Verify the search backend works

Quick test: reverse-search a known AI image (your OpenAI PNG) and see
what Google returns. No re-analysis yet, just the raw matches.

In [3]:
from pathlib import Path
from ai_image_id.web_provenance import _reverse_search, _classify_domain
from google.colab import files

print("Upload an AI image to search for (your ChatGPT PNG is ideal):")
up = files.upload()
test_img = Path(next(iter(up)))

search = _reverse_search(test_img, max_results=10)
print(f"\nGoogle labels: {search.best_guess_labels}")
print(f"found {len(search.matches)} matches:\n")
for i, m in enumerate(search.matches):
    dtype = _classify_domain(m.domain)
    tag = " ← AI gallery" if dtype == "ai_gallery" else (" ← stock photo" if dtype == "stock_photo" else "")
    print(f"  {i+1}. {m.domain}{tag}")
    print(f"     {m.url[:100]}")

Upload an AI image to search for (your ChatGPT PNG is ideal):


Saving pope-francis-ai-jacket.jpg-295135-scaled.webp to pope-francis-ai-jacket.jpg-295135-scaled (1).webp

Google labels: ['deepfake pope']
found 20 matches:

  1. cdn.unitycms.io
     https://cdn.unitycms.io/images/1j51J_oh4mP8sakEtzQaeV.jpg
  2. preview.redd.it
     https://preview.redd.it/vauban-heirloom-official-concept-for-the-future-jacket-v0-dj86jd6r310f1.jpeg
  3. i.redd.it
     https://i.redd.it/dj86jd6r310f1.jpeg
  4. aperture.org
     https://aperture.org/wp-content/uploads/2025/01/257MAG445_Ritchin_HR.jpg
  5. elnacional.cat
     https://www.elnacional.cat/uploads/s1/40/24/64/94/papa-francesc-imatge-fake-anorak.jpeg
  6. foad.univ-lyon1.fr
     https://foad.univ-lyon1.fr/pluginfile.php/1/core_h5p/content/1468/images/file-67fd0bbfee105.jpg
  7. media-assets.vanityfair.it
     https://media-assets.vanityfair.it/photos/64217844f3341d2f14f83008/master/pass/SEI_149760389.jpeg
  8. media.vanityfair.fr
     https://media.vanityfair.fr/photos/6421b571da470a4247714880/master/w_3940%

## §2 — The retry loop: can M5 recover provenance?

The real test: take an image that's been stripped of all metadata (M1 silent,
M2 silent), run the pipeline **without M5** (baseline), then **with M5**
(does it find an upstream copy with intact provenance and inherit the verdict?).

For this to work, the original needs to exist somewhere on the web with its
metadata intact — upload your OpenAI PNG to a public URL first (GitHub
issue comment, public Drive link, or any image host that preserves metadata),
then strip it and feed the stripped version to the pipeline.

In [4]:
import shutil, subprocess
from ai_image_id.main import analyze_image
from ai_image_id.web_provenance import trace_provenance

# Step 1: Create a stripped version of the uploaded image
stripped = Path("/content/stripped_test.jpg")
from PIL import Image
im = Image.open(test_img).convert("RGB")
im.save(stripped, quality=85)  # re-encode as JPEG: kills C2PA, kills watermarks

print("── Baseline: pipeline WITHOUT M5 ──")
r_base = analyze_image(stripped)
print(f"verdict: {r_base.ai_verdict.value} ({r_base.confidence})")
print(f"notes: {r_base.notes}\n")

print("── Now: M5 reverse search on the stripped image ──")
web = trace_provenance(stripped)
print(f"searched: {web['searched']}")
print(f"matches: {web['matches_found']}")
print(f"domains: {web['domains'][:5]}")
print(f"AI gallery match: {web['ai_gallery_match']}")
print(f"AI context match: {web.get('ai_context_match', False)}")
print(f"best guess labels: {web.get('best_guess_labels', [])}")
print(f"upstream verdict: {web['upstream_verdict']}")
print(f"chain: {web['chain']}")
print(f"notes: {web['notes']}")

── Baseline: pipeline WITHOUT M5 ──


preprocessor_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 44.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

verdict: inconclusive (0.5)
notes: ['no AI-indicative signal found', 'absence of watermarks/metadata is non-evidence, not proof of human origin']

── Now: M5 reverse search on the stripped image ──
searched: True
matches: 20
domains: ['elnacional.cat', 'theconversation.com', 'i.redd.it', 'cnn.com', 'sustainabilitymag.lu']
AI gallery match: False
AI context match: True
best guess labels: ['deepfake pope']
upstream verdict: None
chain: []
notes: ["web context suggests AI origin (9 signals): google label: 'deepfake pope'; URL at time.com; URL at cnn.com; title: 'The disturbing trend of state media use of deepfakes'; title: 'Be careful for AI deep fakes of Pope Leo XIV. People have been ...'"]


## §3 — Wired into the pipeline: before/after comparison

Now test with M5 integrated into `analyze_image`. The stripped image should
go from `inconclusive` (no metadata, no watermark) to inheriting a stronger
verdict from the upstream copy — if one exists on the web with intact
provenance.

In [8]:
# Wire M5 into the pipeline: re-analyze with web provenance enabled
def analyze_with_m5(path, detector_ckpt=None):
    """analyze_image + M5 retry loop on inconclusive results."""
    result = analyze_image(str(path), detector_ckpt=detector_ckpt)

    if result.ai_verdict.value == "inconclusive":
        web = trace_provenance(str(path), detector_ckpt=detector_ckpt)

        if web.get("upstream_verdict") in ("verified", "likely"):
            result.notes.append(
                f"M5: upstream copy at {web['upstream_url']} → "
                f"{web['upstream_verdict']}"
            )
            result.notes.extend(web["notes"])
            from ai_image_id.schema import Verdict
            result.ai_verdict = Verdict(web["upstream_verdict"])
            result.confidence = (
                web["chain"][-1]["confidence"] if web["chain"] else 0.75
            )
        elif web.get("ai_context_match"):
            result.notes.extend(web["notes"])
            from ai_image_id.schema import Verdict
            result.ai_verdict = Verdict.LIKELY
            result.confidence = 0.7
        elif web["ai_gallery_match"]:
            result.notes.append(
                f"M5: image found on AI gallery domains "
                f"({', '.join(web['domains'][:3])})"
            )
        elif web["searched"]:
            result.notes.append(
                f"M5: {web['matches_found']} web matches, "
                f"none yielded stronger evidence"
            )

    return result

print("── Pipeline + M5 on the stripped image ──")
r_m5 = analyze_with_m5(stripped)
print(f"verdict: {r_m5.ai_verdict.value} ({r_m5.confidence})")
print(f"notes: {r_m5.notes}")
print()
print(f"── Comparison ──")
print(f"  without M5: {r_base.ai_verdict.value} ({r_base.confidence})")
print(f"  with M5:    {r_m5.ai_verdict.value} ({r_m5.confidence})")

── Pipeline + M5 on the stripped image ──
verdict: likely (0.7)
notes: ['no AI-indicative signal found', 'absence of watermarks/metadata is non-evidence, not proof of human origin', "web context suggests AI origin (9 signals): google label: 'deepfake pope'; URL at time.com; URL at cnn.com; title: 'The disturbing trend of state media use of deepfakes'; title: 'Be careful for AI deep fakes of Pope Leo XIV. People have been ...'"]

── Comparison ──
  without M5: inconclusive (0.5)
  with M5:    likely (0.7)


## §4 — Domain classification: AI galleries vs. stock photos

Upload various images and see what domains Google associates with them.
The domain classification alone (no re-analysis) is cheap, fast signal:
an image whose only web matches are AI gallery domains is contextually
different from one matching stock photo sites.

Upload a mix: AI-generated images, personal photos, stock photos.

In [9]:
from google.colab import files

print("Upload images to classify by web presence:")
up = files.upload()
print()

for name in sorted(up):
    path = Path(name)
    matches = _reverse_search(path, max_results=10)
    domains = list({m.domain for m in matches if m.domain})
    ai_domains = [d for d in domains if _classify_domain(d) == "ai_gallery"]
    stock_domains = [d for d in domains if _classify_domain(d) == "stock_photo"]

    print(f"┌─ {name}")
    print(f"│ matches: {len(matches)}")
    print(f"│ domains: {', '.join(domains[:5])}")
    if ai_domains:    print(f"│ ⚠ AI galleries: {', '.join(ai_domains)}")
    if stock_domains: print(f"│ ✓ stock photos: {', '.join(stock_domains)}")
    if not matches:   print(f"│ no web presence found")
    print(f"└")
    print()

Upload images to classify by web presence:


Saving pope-francis-ai-jacket.jpg-295135-scaled.webp to pope-francis-ai-jacket.jpg-295135-scaled (3).webp



TypeError: 'SearchResult' object is not iterable

## §5 — Transport resilience: can M5 find the original after transforms?

Same 7 hops as the M1/M2/M4 matrices. The question: does Google's reverse
search find the original image after screenshot / crop / recompression?
If yes, M5 can trace provenance even on heavily degraded copies.

This tests the *search* step, not the re-analysis — we're measuring
whether Google Vision recognizes a transformed image as a match.

In [ ]:
import shutil, subprocess

print("Using the image from §1:", test_img.name)
im = Image.open(test_img).convert("RGB")
HOP = Path("/content/hops_m5"); HOP.mkdir(exist_ok=True)

hops = {"0-original": test_img}
p = HOP/"1-resave.jpg";     im.save(p, quality=92);  hops["1-resave-jpg"] = p
p = HOP/"2-screenshot.png"; im.save(p);               hops["2-screenshot"] = p
p = HOP/"3-messenger.jpg";  im.resize((im.width//2, im.height//2)).save(p, quality=70)
hops["3-messenger"] = p
p = HOP/("4-strip"+test_img.suffix); shutil.copy(test_img, p)
subprocess.run(["exiftool","-overwrite_original","-all=",str(p)], capture_output=True)
hops["4-exiftool-strip"] = p
p = HOP/"5-crop.jpg"; im.crop((50,50,im.width-50,im.height-50)).save(p, quality=92)
hops["5-crop"] = p
p = HOP/"6-pil-reencode.png"; im.save(p)
hops["6-pil-reencode"] = p

print(f"\n{'transport':<20} {'matches':>8} {'found original?':>16}")
print("─" * 48)
for name, path in hops.items():
    matches = _reverse_search(path, max_results=5)
    # Check if any match points back to the original domain
    orig_domain = None
    if matches:
        orig_domain = matches[0].domain
    found = len(matches) > 0
    print(f"{name:<20} {len(matches):>8} {'✓' if found else '·':>16}")

### Reading the M5 transport matrix

Unlike M1/M2/M4 which analyze the *local file*, M5 searches the *web*.
Its resilience depends on Google Vision's perceptual matching, not our code.
Expected: Google Vision is quite robust to JPEG recompression and light
cropping (it uses deep features, not pixel-exact matching), but aggressive
resize (messenger hop) may break the match. If all 7 hops return matches,
M5 is the most transport-resilient module — though it requires both web
connectivity and the original to exist online.

## §6 — The full pipeline: M1 × M2 × M4 × M5

The complete composite matrix. For each transport, which modules contribute?

| transport | C2PA (M1) | TrustMark (M2) | Classifier (M4) | Web search (M5) |
|---|---|---|---|---|
| original | ✓ | ✓ | 0.984 | ✓ (if published) |
| re-save | · | ✓ | 0.984 | ✓? |
| screenshot | · | ✓ | 0.984 | ✓? |
| messenger | · | · | 0.979 | ?  |
| exiftool-strip | ✓ | ✓ | 0.984 | ✓? |
| crop | · | ✓ | 0.992 | ?  |
| re-encode | · | ✓ | 0.984 | ✓? |

M5 adds a new dimension: it doesn't analyze the image at all — it asks
whether the internet has seen it before. A module that's orthogonal to
all the others, dependent on the image having been published, but capable
of recovering provenance that every other module lost.